In [ ]:
import pandas as pd
import numpy as np

# 1. read the csv file
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

# 2. Fix type issues
print("\nBefore conversion, TotalCharges type is:", df['TotalCharges'].dtype)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("After conversion, TotalCharges type is:", df['TotalCharges'].dtype)

# 3. Handle missing values created by coercion
null_count = df['TotalCharges'].isnull().sum()
print(f"\nFound {null_count} missing values in TotalCharges column.")
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 4. Analyze class balance
print("\nDistribution of Target Variable (Churn):")
print(df['Churn'].value_counts(normalize=True) * 100)

In [ ]:
# copy of the cleaned dataframe to modify
df_encoded = df.copy()

# 1. customerID column (it holds no predictive power)
df_encoded = df_encoded.drop(columns=['customerID'])

# 2. Convert Binary Text Columns (Yes/No) to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0})

# Also map Gender to 1/0 (Male=1, Female=0)
df_encoded['gender'] = df_encoded['gender'].map({'Male': 1, 'Female': 0})

# 3. This splits options like 'InternetService' into separate columns of 1s and 0s
categorical_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
                    'Contract', 'PaymentMethod']

df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, dtype=int)

# 4. Inspect the results
print("Original Phase 1 Dimensions:", df.shape)
print("New Encoded Phase 2 Dimensions:", df_encoded.shape)
print("\nFirst 3 rows of your fully numerical dataset:")
print(df_encoded.head(3))

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# 1. Separate the "Answers" (y) from the "Questions" (X)
# X is everything EXCEPT the Churn column
X = df_encoded.drop(columns=['Churn'])
# y is ONLY the Churn column
y = df_encoded['Churn']

# 2. Split the data into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("--- BEFORE SMOTE ---")
print(y_train.value_counts())

# 3. Apply SMOTE to fix the imbalance
# CRITICAL RULE: We ONLY apply SMOTE to the training data. 
# We never fake data in the test set, because the test set must reflect reality!
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\n--- AFTER SMOTE ---")
print(y_train_smote.value_counts())
print(f"\nNew Training Data Shape: {X_train_smote.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Initialize the algorithm
# random_state ensures we get the exact same results every time we run it
rf_model = RandomForestClassifier(random_state=42)

# 2. Train (Fit) the model using the BALANCED data from SMOTE
print("Training Random Forest model... (This might take a few seconds)")
rf_model.fit(X_train_smote, y_train_smote)

# 3. Test the model by making predictions on the UNSEEN test data
# Notice we use X_test here, NOT the smote data! The test set must be real data.
y_pred = rf_model.predict(X_test)

# 4. Evaluate the results
print("\n--- Model Accuracy ---")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("--- Confusion Matrix ---")
print("[[True Negatives  False Positives]")
print(" [False Negatives True Positives]]")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
# '0' is No Churn, '1' is Churn
print(classification_report(y_test, y_pred))


In [ ]:
# Phase 5: Threshold Optimization as recall was 0.53 (53%) which is not much favourible

# 1. Instead of getting hard Yes/No predictions, we ask the model for the exact probabilities.
# rf_model.predict_proba returns two columns: [Probability of Stay, Probability of Churn]
# We only want the second column (Probability of Churn), which is index [[:, 1]]
y_pred_probabilities = rf_model.predict_proba(X_test)[:, 1]

# 2. Set our new custom threshold (Lowering it to 30%)
custom_threshold = 0.30

# 3. Create new predictions based on our custom threshold
# If the probability is greater than 0.30, mark as 1 (Churn), else 0 (Stay)
y_pred_custom = (y_pred_probabilities >= custom_threshold).astype(int)

# 4. Evaluate the new results
print(f"--- Results with custom threshold: {custom_threshold} ---")
print("\nNew Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_custom))

print("\nNew Classification Report:")
print(classification_report(y_test, y_pred_custom))

In [ ]:
import joblib

# Saving the trained model to a file
joblib.dump(rf_model, 'churn_model.pkl')
print("Model saved successfully as 'churn_model.pkl'")

# 2. Save the exact list of column names 
# (Crucial for later, so the web app knows exactly what 46 columns the model expects)
import json
with open('model_columns.json', 'w') as f:
    json.dump(list(X_train_smote.columns), f)
    
print("Column mapping saved successfully as 'model_columns.json'")